In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from tqdm.auto import tqdm

In [4]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 12.1


device(type='cuda')

In [6]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [7]:
# ============================================================
# 3. LOAD GENDER PERSPECTIVE DATASET
# ============================================================

DATA_PATH = "../data/processed/gender_perspective_hatespeech_softlabels_min2.csv"

gender_df = pd.read_csv(DATA_PATH)

print("Gender perspective dataset:", gender_df.shape)
print("Unique comments:", gender_df["comment_id"].nunique())
print("Perspectives:", gender_df["perspective"].unique())

gender_df.head()

Gender perspective dataset: (8089, 12)
Unique comments: 7582
Perspectives: ['women' 'men' 'non_binary' 'prefer_not_to_say']


,comment_id,perspective,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,annotation_count,split,text_clean,perspective_type,input_text,entropy,distribution_pattern
0,10,women,0.333333,0.0,0.666667,3,train,"They'll come back in your plan, also. Plus we ...",annotator_gender_group,"[WOMEN] They'll come back in your plan, also. ...",0.918296,"(0.333, 0.0, 0.667)"
1,12,women,0.500000,0.0,0.500000,2,train,She ugly anyways,annotator_gender_group,[WOMEN] She ugly anyways,1.000000,"(0.5, 0.0, 0.5)"
2,19,men,1.000000,0.0,0.000000,2,train,Won't happen. Birth rates are down. Women want...,annotator_gender_group,[MEN] Won't happen. Birth rates are down. Wome...,-0.000000,"(1.0, 0.0, 0.0)"
3,22,men,0.500000,0.0,0.500000,2,train,The guys are one thing but then you have a wom...,annotator_gender_group,[MEN] The guys are one thing but then you have...,1.000000,"(0.5, 0.0, 0.5)"
4,26,men,0.500000,0.0,0.500000,2,train,I'd love to rip those panties off and shove my...,annotator_gender_group,[MEN] I'd love to rip those panties off and sh...,1.000000,"(0.5, 0.0, 0.5)"


In [8]:
# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_cols = [
    "comment_id",
    "perspective",
    "input_text",
    "split",
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob",
    "annotation_count",
    "entropy"
]

missing_cols = [col for col in required_cols if col not in gender_df.columns]

print("Missing columns:", missing_cols)

if len(missing_cols) == 0:
    print("All required columns are present.")

Missing columns: []
All required columns are present.


In [9]:
# ============================================================
# 5. LABEL COLUMNS AND PERSPECTIVES
# ============================================================

LABELS = [0, 1, 2]

label_cols = [
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob"
]

ALLOWED_GENDER_PERSPECTIVES = [
    "women",
    "men",
    "non_binary",
    "prefer_not_to_say"
]

gender_df = gender_df[
    gender_df["perspective"].isin(ALLOWED_GENDER_PERSPECTIVES)
].copy()

print(gender_df["perspective"].value_counts())

perspective
women                4850
men                  3200
non_binary             28
prefer_not_to_say      11
Name: count, dtype: int64


In [10]:
# ============================================================
# 6. DATASET SUMMARY
# ============================================================

perspective_summary = (
    gender_df
    .groupby("perspective")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotations=("annotation_count", "mean"),
        median_annotations=("annotation_count", "median"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2)),
        nonzero_entropy_percent=("entropy", lambda x: round((x > 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values("examples", ascending=False)
)

display(perspective_summary)

split_summary = (
    gender_df
    .groupby("split")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotation_count=("annotation_count", "mean"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2)),
        unique_perspectives=("perspective", "nunique")
    )
    .reset_index()
)

display(split_summary)

perspective_split_summary = (
    gender_df
    .groupby(["split", "perspective"])
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotation_count=("annotation_count", "mean"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values(["split", "examples"], ascending=[True, False])
)

display(perspective_split_summary)

,perspective,examples,unique_comments,mean_annotations,median_annotations,mean_entropy,zero_entropy_percent,nonzero_entropy_percent
3,women,4850,4850,3.785361,2.0,0.381104,61.57,38.43
0,men,3200,3200,3.879688,2.0,0.362966,63.22,36.78
1,non_binary,28,28,3.607143,3.0,0.481488,46.43,53.57
2,prefer_not_to_say,11,11,3.181818,2.0,0.306450,72.73,27.27


,split,examples,unique_comments,mean_annotation_count,mean_entropy,zero_entropy_percent,unique_perspectives
0,test,1224,1140,4.035131,0.376000,62.09,4
1,train,5672,5316,3.783850,0.374394,62.17,4
2,validation,1193,1126,3.779547,0.371255,62.36,3


,split,perspective,examples,unique_comments,mean_annotation_count,mean_entropy,zero_entropy_percent
3,test,women,710,710,4.059155,0.376094,62.25
0,test,men,507,507,3.998028,0.377073,61.74
1,test,non_binary,4,4,5.250000,0.162506,75.00
2,test,prefer_not_to_say,3,3,3.000000,0.456984,66.67
7,train,women,3392,3392,3.757075,0.379244,61.73
4,train,men,2251,2251,3.831186,0.366411,62.91
5,train,non_binary,21,21,3.238095,0.494293,47.62
6,train,prefer_not_to_say,8,8,3.250000,0.250000,75.00
10,validation,women,748,748,3.653743,0.394293,60.16
8,validation,men,442,442,3.990950,0.329240,66.52


In [11]:

train_df = gender_df[gender_df["split"] == "train"].copy()
val_df = gender_df[gender_df["split"] == "validation"].copy()
test_df = gender_df[gender_df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain perspectives:")
print(train_df["perspective"].value_counts())

print("\nValidation perspectives:")
print(val_df["perspective"].value_counts())

print("\nTest perspectives:")
print(test_df["perspective"].value_counts())

Train: (5672, 12)
Validation: (1193, 12)
Test: (1224, 12)

Train perspectives:
perspective
women                3392
men                  2251
non_binary             21
prefer_not_to_say       8
Name: count, dtype: int64

Validation perspectives:
perspective
women         748
men           442
non_binary      3
Name: count, dtype: int64

Test perspectives:
perspective
women                710
men                  507
non_binary             4
prefer_not_to_say      3
Name: count, dtype: int64


In [13]:
hf_cols = [
    "comment_id",
    "perspective",
    "input_text",
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob",
    "annotation_count",
    "entropy"
]

train_dataset = Dataset.from_pandas(train_df[hf_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[hf_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[hf_cols].reset_index(drop=True))

In [14]:
model_name = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

special_tokens = {
    "additional_special_tokens": [
        "[WOMEN]",
        "[MEN]",
        "[NON_BINARY]",
        "[PREFER_NOT_TO_SAY]"
    ]
}

num_added = tokenizer.add_special_tokens(special_tokens)

print("Added special tokens:", num_added)
print("Tokenizer size:", len(tokenizer))

Added special tokens: 4
Tokenizer size: 50269


In [15]:
def tokenize(batch):
    return tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5672 [00:00<?, ? examples/s]

Map:   0%|          | 0/1193 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

In [16]:
def add_labels(batch):
    n = len(batch["input_text"])

    batch["labels"] = [
        [
            float(batch["hatespeech_0_prob"][i]),
            float(batch["hatespeech_1_prob"][i]),
            float(batch["hatespeech_2_prob"][i])
        ]
        for i in range(n)
    ]

    return batch

train_dataset = train_dataset.map(add_labels, batched=True)
val_dataset = val_dataset.map(add_labels, batched=True)
test_dataset = test_dataset.map(add_labels, batched=True)

Map:   0%|          | 0/5672 [00:00<?, ? examples/s]

Map:   0%|          | 0/1193 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

In [17]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [18]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [19]:
class GenderPerspectiveLeWiDiModel(nn.Module):
    def __init__(self, model_name, tokenizer_size, dropout_prob=0.35, freeze_layers=4):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.resize_token_embeddings(tokenizer_size)

        hidden_size = self.encoder.config.hidden_size

        if freeze_layers > 0:
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

            for layer in self.encoder.encoder.layer[:freeze_layers]:
                for param in layer.parameters():
                    param.requires_grad = False

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)

        logits = self.classifier(pooled)
        return logits

In [20]:
model = GenderPerspectiveLeWiDiModel(
    model_name=model_name,
    tokenizer_size=len(tokenizer),
    dropout_prob=0.35,
    freeze_layers=4
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable_params)
print("Total params:", total_params)
print("Trainable percent:", round(trainable_params / total_params * 100, 2))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 57295875
Total params: 124651011
Trainable percent: 45.97


In [21]:
def compute_loss(model, batch):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].float().to(device)

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    loss = torch.nn.functional.kl_div(
        log_probs,
        labels,
        reduction="batchmean"
    )

    return loss

In [22]:
def distribution_metrics(probs, labels):
    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    hard_accuracy = (probs.argmax(axis=1) == labels.argmax(axis=1)).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    if np.std(pred_entropy) == 0 or np.std(gold_entropy) == 0:
        entropy_corr = np.nan
    else:
        entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    entropy_mae = np.mean(np.abs(pred_entropy - gold_entropy))

    return {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr),
        "entropy_mae": float(entropy_mae),
    }

In [23]:
def predict_loader(model, loader):
    model.eval()

    all_probs = []
    all_labels = []
    losses = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            labels = batch["labels"].float()
            loss = compute_loss(model, batch)

            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            probs = torch.nn.functional.softmax(logits, dim=-1)

            losses.append(loss.item())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    probs = np.vstack(all_probs)
    labels = np.vstack(all_labels)

    return probs, labels, float(np.mean(losses))

In [24]:
def evaluate_overall(model, loader):
    probs, labels, loss = predict_loader(model, loader)
    metrics = distribution_metrics(probs, labels)
    metrics["eval_loss"] = loss
    return metrics

In [25]:
def evaluate_by_perspective(model, loader, original_df):
    probs, labels, _ = predict_loader(model, loader)

    out = original_df.reset_index(drop=True).copy()

    out["pred_0"] = probs[:, 0]
    out["pred_1"] = probs[:, 1]
    out["pred_2"] = probs[:, 2]

    out["gold_0"] = labels[:, 0]
    out["gold_1"] = labels[:, 1]
    out["gold_2"] = labels[:, 2]

    rows = []

    for perspective, group_df in out.groupby("perspective"):
        group_probs = group_df[["pred_0", "pred_1", "pred_2"]].values
        group_labels = group_df[["gold_0", "gold_1", "gold_2"]].values

        metrics = distribution_metrics(group_probs, group_labels)

        row = {
            "perspective": perspective,
            "n": len(group_df),
            "mean_annotation_count": group_df["annotation_count"].mean(),
            "mean_entropy": group_df["entropy"].mean(),
        }

        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("n", ascending=False), out

In [36]:
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.10
NUM_EPOCHS = 10
PATIENCE = 3

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [37]:
output_dir = "../models/roberta_gender_perspective_conditioned_lewidi_1"
best_model_dir = os.path.join(output_dir, "best_model")

os.makedirs(best_model_dir, exist_ok=True)

history_rows = []

best_val_js = float("inf")
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()

    train_losses = []

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")

    for batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        loss = compute_loss(model, batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()

        train_losses.append(loss.item())

        progress.set_postfix({
            "train_loss": np.mean(train_losses)
        })

    val_metrics = evaluate_overall(model, val_loader)
    val_perspective_metrics, _ = evaluate_by_perspective(
        model,
        val_loader,
        val_df
    )

    print("\n" + "=" * 80)
    print(f"Epoch {epoch}")
    print(f"Train loss: {np.mean(train_losses):.6f}")
    print("Validation overall:")
    for k, v in val_metrics.items():
        print(f"{k}: {v}")

    print("\nValidation by perspective:")
    print(val_perspective_metrics.to_string(index=False))
    print("=" * 80 + "\n")

    history_row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
    }

    for k, v in val_metrics.items():
        history_row[f"val_{k}"] = v

    history_rows.append(history_row)

    current_val_js = val_metrics["js_divergence"]

    if current_val_js < best_val_js:
        best_val_js = current_val_js
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            os.path.join(best_model_dir, "pytorch_model.bin")
        )

        tokenizer.save_pretrained(best_model_dir)

        print(f"New best model saved. Best validation JS: {best_val_js:.6f}")

    else:
        epochs_without_improvement += 1

        print(f"No improvement. Patience: {epochs_without_improvement}/{PATIENCE}")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    torch.cuda.empty_cache()

Epoch 1/10:   0%|          | 0/709 [00:00<?, ?it/s]


Epoch 1
Train loss: 0.350824
Validation overall:
kl_divergence: 0.4391670227050781
js_divergence: 0.10313868522644043
hard_accuracy: 0.7770326906957251
entropy_correlation: 0.3412141385683018
entropy_mae: 0.49972179532051086
eval_loss: 0.4377290324370066

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 748               3.653743      0.394293       0.465583       0.108135       0.778075             0.327611     0.505451
        men 442               3.990950      0.329240       0.386271       0.093895       0.778281             0.364667     0.490883
 non_binary   3               4.000000      0.817167       1.645962       0.219220       0.333333            -0.814779     0.373523

New best model saved. Best validation JS: 0.103139


Epoch 2/10:   0%|          | 0/709 [00:00<?, ?it/s]


Epoch 2
Train loss: 0.334326
Validation overall:
kl_divergence: 0.4515545964241028
js_divergence: 0.10182584822177887
hard_accuracy: 0.7912824811399832
entropy_correlation: 0.34111199990920404
entropy_mae: 0.49017053842544556
eval_loss: 0.4501095853249232

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 748               3.653743      0.394293       0.480894       0.107539       0.791444             0.327626     0.499226
        men 442               3.990950      0.329240       0.393340       0.091385       0.794118             0.363831     0.475466
 non_binary   3               4.000000      0.817167       1.713181       0.215488       0.333333            -0.788778     0.398898

New best model saved. Best validation JS: 0.101826


Epoch 3/10:   0%|          | 0/709 [00:00<?, ?it/s]


Epoch 3
Train loss: 0.308408
Validation overall:
kl_divergence: 0.46463221311569214
js_divergence: 0.10672535747289658
hard_accuracy: 0.7736797988264879
entropy_correlation: 0.31683482566709525
entropy_mae: 0.5133392810821533
eval_loss: 0.46303805033365886

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 748               3.653743      0.394293       0.493845       0.112358       0.775401             0.295610     0.521916
        men 442               3.990950      0.329240       0.406387       0.096455       0.773756             0.354963     0.500218
 non_binary   3               4.000000      0.817167       1.762422       0.215371       0.333333            -0.814078     0.308009

No improvement. Patience: 1/3


Epoch 4/10:   0%|          | 0/709 [00:00<?, ?it/s]


Epoch 4
Train loss: 0.285024
Validation overall:
kl_divergence: 0.47565340995788574
js_divergence: 0.10966534912586212
hard_accuracy: 0.7778709136630344
entropy_correlation: 0.2992563806932339
entropy_mae: 0.5302591919898987
eval_loss: 0.47410740117232003

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 748               3.653743      0.394293       0.498964       0.114429       0.780749             0.285809     0.542851
        men 442               3.990950      0.329240       0.427079       0.100826       0.776018             0.320339     0.510337
 non_binary   3               4.000000      0.817167       1.820124       0.224139       0.333333            -0.845716     0.325996

No improvement. Patience: 2/3


Epoch 5/10:   0%|          | 0/709 [00:00<?, ?it/s]


Epoch 5
Train loss: 0.264163
Validation overall:
kl_divergence: 0.5195323824882507
js_divergence: 0.10944078117609024
hard_accuracy: 0.7937971500419112
entropy_correlation: 0.30272662220318447
entropy_mae: 0.5090377926826477
eval_loss: 0.5175258864959081

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 748               3.653743      0.394293       0.548325       0.114618       0.795455             0.294079     0.517784
        men 442               3.990950      0.329240       0.462446       0.099985       0.794118             0.315988     0.494617
 non_binary   3               4.000000      0.817167       1.751256       0.211858       0.333333            -0.736473     0.452957

No improvement. Patience: 3/3
Early stopping triggered.


In [38]:
model.load_state_dict(
    torch.load(
        os.path.join(best_model_dir, "pytorch_model.bin"),
        map_location=device
    )
)

model.to(device)

val_results = evaluate_overall(model, val_loader)
test_results = evaluate_overall(model, test_loader)

val_perspective_results, val_pred_df = evaluate_by_perspective(
    model,
    val_loader,
    val_df
)

test_perspective_results, test_pred_df = evaluate_by_perspective(
    model,
    test_loader,
    test_df
)

print("Final validation overall:")
print(val_results)

print("\nFinal validation by perspective:")
display(val_perspective_results)

print("\nFinal test overall:")
print(test_results)

print("\nFinal test by perspective:")
display(test_perspective_results)

/tmp/ipykernel_64543/812031077.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Final validation overall:
{'kl_divergence': 0.4515545964241028, 'js_divergence': 0.10182584822177887, 'hard_accuracy': 0.7912824811399832, 'entropy_correlation': 0.34111199990920404, 'entropy_mae': 0.49017053842544556, 'eval_loss': 0.4501095853249232}

Final validation by perspective:


,perspective,n,mean_annotation_count,mean_entropy,kl_divergence,js_divergence,hard_accuracy,entropy_correlation,entropy_mae
2,women,748,3.653743,0.394293,0.480894,0.107539,0.791444,0.327626,0.499226
0,men,442,3.990950,0.329240,0.393340,0.091385,0.794118,0.363831,0.475466
1,non_binary,3,4.000000,0.817167,1.713181,0.215488,0.333333,-0.788778,0.398898



Final test overall:
{'kl_divergence': 0.45762285590171814, 'js_divergence': 0.10560687631368637, 'hard_accuracy': 0.7712418300653595, 'entropy_correlation': 0.360225831163701, 'entropy_mae': 0.4804607629776001, 'eval_loss': 0.4569313435198425}

Final test by perspective:


,perspective,n,mean_annotation_count,mean_entropy,kl_divergence,js_divergence,hard_accuracy,entropy_correlation,entropy_mae
3,women,710,4.059155,0.376094,0.468197,0.107968,0.773239,0.345452,0.489696
0,men,507,3.998028,0.377073,0.447838,0.103374,0.765286,0.378897,0.469631
1,non_binary,4,5.250000,0.162506,0.082411,0.026124,1.000000,0.617036,0.241498
2,prefer_not_to_say,3,3.000000,0.456984,0.108905,0.030195,1.000000,0.560930,0.443493


In [39]:
pyevall_dir = "../results/pyevall/gender_perspective_conditioned_lewidi_1"
os.makedirs(pyevall_dir, exist_ok=True)

gold_records = []
pred_records = []

for _, row in test_pred_df.iterrows():
    record_id = str(row["comment_id"]) + "_" + str(row["perspective"])

    gold_records.append({
        "test_case": "gender_perspective_hatespeech",
        "id": record_id,
        "value": {
            "0": float(row["gold_0"]),
            "1": float(row["gold_1"]),
            "2": float(row["gold_2"])
        }
    })

    pred_records.append({
        "test_case": "gender_perspective_hatespeech",
        "id": record_id,
        "value": {
            "0": float(row["pred_0"]),
            "1": float(row["pred_1"]),
            "2": float(row["pred_2"])
        }
    })

gold_path = os.path.join(pyevall_dir, "gender_perspective_hatespeech_gold.json")
pred_path = os.path.join(pyevall_dir, "gender_perspective_hatespeech_pred.json")

with open(gold_path, "w", encoding="utf-8") as f:
    json.dump(gold_records, f, indent=2)

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(pred_records, f, indent=2)

print("Gold:", gold_path)
print("Pred:", pred_path)

Gold: ../results/pyevall/gender_perspective_conditioned_lewidi_1/gender_perspective_hatespeech_gold.json
Pred: ../results/pyevall/gender_perspective_conditioned_lewidi_1/gender_perspective_hatespeech_pred.json


In [40]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils

params = {
    PyEvALLUtils.PARAM_REPORT: PyEvALLUtils.PARAM_OPTION_REPORT_EMBEDDED
}

evaluator = PyEvALLEvaluation()

pyevall_report = evaluator.evaluate(
    pred_path,
    gold_path,
    [
        "CrossEntropy",
        "ICMSoft",
        "ICMSoftNorm"
    ],
    **params
)

pyevall_data = pyevall_report.report if hasattr(pyevall_report, "report") else pyevall_report.__dict__
pyevall_data

2026-06-02 20:03:43,135 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['CrossEntropy', 'ICMSoft', 'ICMSoftNorm']
2026-06-02 20:03:43,376 - pyevall.metrics.metrics - INFO -             evaluate() - Executing Cross Entropy evaluation method for testcase gender_perspective_hatespeech
2026-06-02 20:03:43,651 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase gender_perspective_hatespeech
2026-06-02 20:03:44,180 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM-Soft Normalized evaluation method for testcase gender_perspective_hatespeech
2026-06-02 20:03:44,181 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase gender_perspective_hatespeech
2026-06-02 20:03:44,694 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase gender_perspective_hatespeech


{'metrics': {'CrossEntropy': {'name': 'Cross Entropy',
   'acronym': 'CE',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'gender_perspective_hatespeech',
      'average': 1.0429603162395125}],
    'average_per_test_case': 1.0429603162395125}},
  'ICMSoft': {'name': 'Information Contrast Model Soft',
   'acronym': 'ICM-Soft',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'gender_perspective_hatespeech',
      'average': -2.1381271651781613}],
    'average_per_test_case': -2.1381271651781613}},
  'ICMSoftNorm': {'name': 'Normalized Information Contrast Model Soft',
   'acronym': 'ICM-Soft-Norm',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'gender_perspective_hatespeech',
      'average': 0.27410612018893676}],
    'average_per_test_case': 0.27410612018893676}}},
 'files': {'gender_perspective_hatespeech_pred.json': {'name': 'gender_perspective_hatesp

In [41]:
pyevall_summary = {}

for metric_name, metric_data in pyevall_data["metrics"].items():
    pyevall_summary[metric_name] = metric_data["results"]["average_per_test_case"]

pyevall_summary

{'CrossEntropy': 1.0429603162395125,
 'ICMSoft': -2.1381271651781613,
 'ICMSoftNorm': 0.27410612018893676}

In [42]:
os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_1_results.txt"
history_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_1_training_history.csv"
test_pred_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_1_test_predictions.csv"
test_perspective_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_1_test_perspective_metrics.csv"

history_df = pd.DataFrame(history_rows)

history_df.to_csv(history_path, index=False)
test_pred_df.to_csv(test_pred_path, index=False)
test_perspective_results.to_csv(test_perspective_path, index=False)

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA GENDER-PERSPECTIVE CONDITIONED LEWIDI MODEL RESULTS\n")
    f.write("=" * 90 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 90 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {output_dir}\n")
    f.write(f"Best validation JS: {best_val_js}\n")
    f.write(f"Epochs completed: {len(history_df)}\n\n")

    f.write("2. DATASET STRATEGY\n")
    f.write("-" * 90 + "\n")
    f.write("Task: perspective-conditioned hatespeech soft-label prediction.\n")
    f.write("Each example is a comment-perspective pair.\n")
    f.write("Input format: [PERSPECTIVE] comment text.\n")
    f.write("Target: perspective-specific hatespeech annotator distribution.\n")
    f.write(f"Minimum annotations per comment-perspective pair: 2\n\n")

    f.write("Gender perspectives included:\n")
    for p in ALLOWED_GENDER_PERSPECTIVES:
        f.write(f"- {p}\n")
    f.write("\n")

    f.write("Perspective summary:\n")
    f.write(perspective_summary.to_string(index=False))
    f.write("\n\n")

    f.write("Split summary:\n")
    f.write(split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("Perspective split summary:\n")
    f.write(perspective_split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("3. MODEL OBJECTIVE\n")
    f.write("-" * 90 + "\n")
    f.write("Architecture: shared RoBERTa encoder with one 3-class distribution head.\n")
    f.write("Perspective is provided as a special input token, not as a prediction target.\n")
    f.write("Loss: KL divergence between predicted and perspective-specific gold distributions.\n\n")

    f.write("4. TRAINING SETUP\n")
    f.write("-" * 90 + "\n")
    f.write(f"Max length: {MAX_LENGTH}\n")
    f.write(f"Train batch size: {BATCH_SIZE}\n")
    f.write(f"Eval batch size: {EVAL_BATCH_SIZE}\n")
    f.write(f"Learning rate: {LEARNING_RATE}\n")
    f.write(f"Weight decay: {WEIGHT_DECAY}\n")
    f.write("Dropout: 0.35\n")
    f.write("Frozen lower RoBERTa layers: 4\n")
    f.write(f"Max epochs: {NUM_EPOCHS}\n")
    f.write(f"Early stopping patience: {PATIENCE}\n\n")

    f.write("5. TRAINING HISTORY\n")
    f.write("-" * 90 + "\n")
    f.write(history_df.to_string(index=False))
    f.write("\n\n")

    f.write("6. FINAL VALIDATION RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in val_results.items():
        f.write(f"{k}: {v}\n")

    f.write("\nValidation by perspective:\n")
    f.write(val_perspective_results.to_string(index=False))
    f.write("\n\n")

    f.write("7. FINAL TEST RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in test_results.items():
        f.write(f"{k}: {v}\n")

    f.write("\nTest by perspective:\n")
    f.write(test_perspective_results.to_string(index=False))
    f.write("\n\n")

    f.write("8. PYEVAL LEWIDI RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in pyevall_summary.items():
        f.write(f"{k}: {v}\n")
    f.write("\n")

    

print("Saved:", results_path)
print(open(results_path, encoding="utf-8").read())

Saved: ../results/tables/roberta_gender_perspective_conditioned_lewidi_1_results.txt
ROBERTA GENDER-PERSPECTIVE CONDITIONED LEWIDI MODEL RESULTS

1. RUN INFORMATION
------------------------------------------------------------------------------------------
Run timestamp: 2026-06-02 20:03:45
Model name: roberta-base
Output directory: ../models/roberta_gender_perspective_conditioned_lewidi_1
Best validation JS: 0.10182584822177887
Epochs completed: 5

2. DATASET STRATEGY
------------------------------------------------------------------------------------------
Task: perspective-conditioned hatespeech soft-label prediction.
Each example is a comment-perspective pair.
Input format: [PERSPECTIVE] comment text.
Target: perspective-specific hatespeech annotator distribution.
Minimum annotations per comment-perspective pair: 2

Gender perspectives included:
- women
- men
- non_binary
- prefer_not_to_say

Perspective summary:
      perspective  examples  unique_comments  mean_annotations  median